# Phase 5 — Feature Engineering

**Project:** Flight Delay Prediction  
**Input:** `D:/project/data/processed/bts_weather_joined.parquet`  
**Output:** `D:/project/data/processed/features_final.parquet`  
**Engine:** PySpark (reading Parquet is fast — no CSV scanning overhead)

### Note on 2020 Data
2020 is intentionally **included** in this dataset. COVID-19 caused a dramatic collapse in flight volume (~60% reduction) and an unusual drop in delay rates due to near-empty aircraft and uncongested airspace. This creates a structural break in the data that a naive model will mislearn from. We handle this by:
- Adding an `is_covid_year` binary flag (1 for 2020, 0 otherwise)
- Adding a `flight_volume_index` feature that captures relative traffic level per airport per month
- Documenting 2020 behavior in EDA comments below
- NOT dropping 2020 — the flag lets the model learn the anomaly rather than being blindsided by it

### Features Being Built
| Group | Features |
|---|---|
| Time | hour-of-day, day-of-week, month, quarter, season, is_weekend, is_holiday_period |
| Weather | severity flags, temp extremes, wind categories, visibility categories, precip categories |
| Airport | hub vs non-hub, historical delay rate per airport, congestion index |
| Flight | route frequency, carrier delay history, distance bucket |
| Anomaly | is_covid_year, flight_volume_index |

---
## 0. Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

PARQUET_IN  = "D:/project/data/processed/bts_weather_joined.parquet"
PARQUET_OUT = "D:/project/data/processed/features_final.parquet"

# Hub airports (15 major) vs non-hub (5 mid-sized)
HUB_AIRPORTS = {
    "ATL", "LAX", "ORD", "DFW", "DEN",
    "JFK", "SFO", "CLT", "LAS", "PHX",
    "MIA", "IAH", "SEA", "EWR", "BOS"
}

# US federal holiday periods — high-traffic, high-delay windows
# Format: (month, day_start, day_end)
HOLIDAY_PERIODS = [
    (1,  1,  2),   # New Year
    (5, 23, 27),   # Memorial Day weekend
    (7,  1,  7),   # Independence Day
    (9,  1,  2),   # Labor Day
    (11, 20, 30),  # Thanksgiving week
    (12, 20, 31),  # Christmas / New Year
]

print("Config ready.")

Config ready.


---
## 1. Start Spark

In [2]:
spark = (
    SparkSession.builder
    .appName("FlightDelayFeatureEngineering")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.ui.showConsoleProgress", "false")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready.")

KeyboardInterrupt: 

---
## 2. Load Parquet

Reading Parquet is fast — no CSV scanning. Spark handles this well even on 16GB RAM.

In [ ]:
df = spark.read.parquet(PARQUET_IN)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")
df.printSchema()

In [ ]:
# Cast FL_DATE to proper date type if stored as string
df = df.withColumn("FL_DATE", F.to_date(F.col("FL_DATE")))
df.select("FL_DATE", "AIRPORT_CODE", "DEP_DEL15", "DepDelayMinutes").show(5)

---
## 3. Time Features

In [ ]:
df = (
    df
    # Basic calendar extracts
    .withColumn("year",          F.year("FL_DATE"))
    .withColumn("month",         F.month("FL_DATE"))
    .withColumn("day_of_month",  F.dayofmonth("FL_DATE"))
    .withColumn("day_of_week",   F.dayofweek("FL_DATE"))   # 1=Sun, 7=Sat
    .withColumn("week_of_year",  F.weekofyear("FL_DATE"))
    .withColumn("quarter",       F.quarter("FL_DATE"))

    # Weekend flag (Sat=7, Sun=1)
    .withColumn("is_weekend",
        F.when(F.dayofweek("FL_DATE").isin(1, 7), 1).otherwise(0)
    )

    # Season (meteorological)
    .withColumn("season",
        F.when(F.month("FL_DATE").isin(12, 1, 2), "Winter")
         .when(F.month("FL_DATE").isin(3, 4, 5),  "Spring")
         .when(F.month("FL_DATE").isin(6, 7, 8),  "Summer")
         .otherwise("Fall")
    )

    # Departure hour bucket from CRSDepTime (format: HHMM as integer)
    # e.g. 1430 → hour 14
    .withColumn("dep_hour",
        F.when(F.col("CRSDepTime").isNotNull(),
            (F.col("CRSDepTime").cast(IntegerType()) / 100).cast(IntegerType())
        ).otherwise(None)
    )

    # Time-of-day bucket
    .withColumn("time_of_day",
        F.when(F.col("dep_hour").between(5,  11), "Morning")
         .when(F.col("dep_hour").between(12, 16), "Afternoon")
         .when(F.col("dep_hour").between(17, 20), "Evening")
         .when(F.col("dep_hour").between(21, 23), "Night")
         .otherwise("RedEye")   # 0-4
    )
)

print("Time features added.")
df.select("FL_DATE", "year", "month", "day_of_week", "season",
          "dep_hour", "time_of_day", "is_weekend").show(5)

In [ ]:
# Holiday period flag
# Build condition dynamically from HOLIDAY_PERIODS list
holiday_condition = F.lit(False)
for (m, d_start, d_end) in HOLIDAY_PERIODS:
    holiday_condition = holiday_condition | (
        (F.col("month") == m) &
        (F.col("day_of_month") >= d_start) &
        (F.col("day_of_month") <= d_end)
    )

df = df.withColumn("is_holiday_period",
    F.when(holiday_condition, 1).otherwise(0)
)

# Verify — Thanksgiving week should show is_holiday_period=1
df.filter((F.col("month") == 11) & (F.col("day_of_month") == 25)) \
  .select("FL_DATE", "month", "day_of_month", "is_holiday_period") \
  .show(3)

---
## 4. COVID / Anomaly Features

**2020 note:** Flight volume dropped ~60% in April–June 2020 and delay rates fell paradoxically
because aircraft were near-empty and airspace was uncongested. Without a flag, a model trained
on 2015–2024 will see 2020 as "good weather = low delays" when the real cause was a global pandemic.
The `is_covid_year` flag lets tree-based models isolate this regime. The `flight_volume_index`
captures the same signal continuously (low volume = unusual conditions).

In [ ]:
# is_covid_year: binary flag for 2020
df = df.withColumn("is_covid_year",
    F.when(F.col("year") == 2020, 1).otherwise(0)
)

# Flight volume index: monthly flight count per airport
# normalized to 0-1 within each airport across all years
# This captures 2020 collapse AND normal seasonal variation
monthly_volume = df.groupBy("AIRPORT_CODE", "year", "month").agg(
    F.count("*").alias("monthly_flights")
)

# Min-max normalize per airport
airport_window = Window.partitionBy("AIRPORT_CODE")
monthly_volume = (
    monthly_volume
    .withColumn("min_vol", F.min("monthly_flights").over(airport_window))
    .withColumn("max_vol", F.max("monthly_flights").over(airport_window))
    .withColumn("flight_volume_index",
        F.round(
            (F.col("monthly_flights") - F.col("min_vol")) /
            (F.col("max_vol") - F.col("min_vol") + F.lit(1e-9)),
        4)
    )
    .select("AIRPORT_CODE", "year", "month", "monthly_flights", "flight_volume_index")
)

# Join back to main df
df = df.join(monthly_volume, on=["AIRPORT_CODE", "year", "month"], how="left")

print("COVID/anomaly features added.")

# Verify — 2020 should show low flight_volume_index
df.groupBy("year").agg(
    F.count("*").alias("flights"),
    F.round(F.avg("flight_volume_index"), 3).alias("avg_vol_index"),
    F.avg("is_covid_year").alias("is_covid")
).orderBy("year").show()

---
## 5. Weather Features

In [ ]:
df = (
    df
    # ── Temperature extremes ──────────────────────────────────
    # Freezing flag — below 0°C increases ice/deicing delays significantly
    .withColumn("is_freezing",
        F.when(F.col("min_temp_c") < 0, 1).otherwise(0)
    )
    # Extreme heat flag — above 38°C (100°F) causes density altitude issues
    .withColumn("is_extreme_heat",
        F.when(F.col("max_temp_c") > 38, 1).otherwise(0)
    )
    # Temperature swing (diurnal range) — large swings can affect equipment
    .withColumn("temp_range_c",
        F.round(F.col("max_temp_c") - F.col("min_temp_c"), 2)
    )

    # ── Wind categories (m/s) ────────────────────────────────
    # ICAO crosswind limits: ~15 m/s = significant, >20 m/s = severe
    .withColumn("wind_category",
        F.when(F.col("max_wind_ms") < 5,  "Calm")
         .when(F.col("max_wind_ms") < 10, "Moderate")
         .when(F.col("max_wind_ms") < 15, "Strong")
         .when(F.col("max_wind_ms") < 20, "Very Strong")
         .otherwise("Severe")
    )
    .withColumn("is_high_wind",
        F.when(F.col("max_wind_ms") >= 15, 1).otherwise(0)
    )

    # ── Visibility categories (km) ───────────────────────────
    # IFR < 5km, LIFR < 1.6km, VFR >= 8km
    .withColumn("visibility_category",
        F.when(F.col("min_visibility_km") < 1.6,  "LIFR")   # Low IFR — worst
         .when(F.col("min_visibility_km") < 5.0,  "IFR")    # Instrument rules
         .when(F.col("min_visibility_km") < 8.0,  "MVFR")   # Marginal VFR
         .otherwise("VFR")                                    # Visual rules — best
    )
    .withColumn("is_low_visibility",
        F.when(F.col("min_visibility_km") < 5.0, 1).otherwise(0)
    )

    # ── Precipitation categories (mm) ────────────────────────
    .withColumn("precip_category",
        F.when(F.col("total_precip_mm").isNull() | (F.col("total_precip_mm") == 0), "None")
         .when(F.col("total_precip_mm") < 2.5,  "Light")
         .when(F.col("total_precip_mm") < 10.0, "Moderate")
         .when(F.col("total_precip_mm") < 50.0, "Heavy")
         .otherwise("Extreme")
    )
    .withColumn("is_precipitation",
        F.when(F.col("total_precip_mm") > 0, 1).otherwise(0)
    )

    # ── Ceiling categories (metres) ──────────────────────────
    # Low ceiling = IFR conditions even if horizontal visibility is ok
    .withColumn("is_low_ceiling",
        F.when(F.col("min_ceiling_m") < 300, 1).otherwise(0)   # < 1000ft
    )

    # ── Composite weather severity score (0-5) ───────────────
    # Simple additive score — useful as a single numeric weather feature
    .withColumn("weather_severity_score",
        F.col("is_freezing").cast(IntegerType()) +
        F.col("is_extreme_heat").cast(IntegerType()) +
        F.col("is_high_wind").cast(IntegerType()) +
        F.col("is_low_visibility").cast(IntegerType()) +
        F.col("is_low_ceiling").cast(IntegerType())
    )
)

print("Weather features added.")
df.select(
    "AIRPORT_CODE", "FL_DATE",
    "is_freezing", "is_high_wind", "wind_category",
    "visibility_category", "precip_category",
    "weather_severity_score"
).show(5)

---
## 6. Airport Features

In [ ]:
# Hub vs non-hub flag
hub_list = list(HUB_AIRPORTS)
df = df.withColumn("is_hub",
    F.when(F.col("AIRPORT_CODE").isin(hub_list), 1).otherwise(0)
)

print("Hub flag added.")
df.groupBy("is_hub").agg(
    F.count("*").alias("flights"),
    F.round(F.avg("DEP_DEL15") * 100, 1).alias("delay_rate_%")
).show()

In [ ]:
# Historical airport delay rate — computed EXCLUDING 2020
# to avoid the COVID anomaly biasing the baseline delay profile per airport
# This is used as a static feature: "how delay-prone is this airport historically"
airport_delay_rate = (
    df
    .filter(F.col("year") != 2020)
    .filter(F.col("DEP_DEL15").isNotNull())
    .groupBy("AIRPORT_CODE")
    .agg(
        F.round(F.avg("DEP_DEL15"), 4).alias("airport_hist_delay_rate"),
        F.round(F.avg("DepDelayMinutes"), 2).alias("airport_hist_avg_delay_mins")
    )
)

df = df.join(airport_delay_rate, on="AIRPORT_CODE", how="left")

print("Historical airport delay rate added (excludes 2020).")
airport_delay_rate.orderBy(F.desc("airport_hist_delay_rate")).show(20)

In [ ]:
# Daily congestion index — how busy is this airport on this specific day
# relative to its own historical average for that month
# High congestion days have higher delay probability independent of weather
daily_traffic = df.groupBy("AIRPORT_CODE", "FL_DATE").agg(
    F.count("*").alias("daily_flight_count")
)

# Average daily traffic per airport per month (baseline)
monthly_avg_traffic = (
    daily_traffic
    .withColumn("month", F.month(F.to_date("FL_DATE")))
    .groupBy("AIRPORT_CODE", "month")
    .agg(F.avg("daily_flight_count").alias("avg_daily_flights_that_month"))
)

daily_traffic = (
    daily_traffic
    .withColumn("month", F.month(F.to_date("FL_DATE")))
    .join(monthly_avg_traffic, on=["AIRPORT_CODE", "month"], how="left")
    .withColumn("congestion_index",
        F.round(
            F.col("daily_flight_count") / (F.col("avg_daily_flights_that_month") + F.lit(1e-9)),
        4)
    )
    .select("AIRPORT_CODE", "FL_DATE", "daily_flight_count", "congestion_index")
)

df = df.join(daily_traffic, on=["AIRPORT_CODE", "FL_DATE"], how="left")

print("Congestion index added.")
df.select("AIRPORT_CODE", "FL_DATE", "daily_flight_count", "congestion_index").show(5)

---
## 7. Flight / Route Features

In [ ]:
# Distance bucket
df = df.withColumn("distance_bucket",
    F.when(F.col("Distance") < 500,  "Short")    # < 500 miles
     .when(F.col("Distance") < 1500, "Medium")   # 500–1500
     .when(F.col("Distance") < 2500, "Long")     # 1500–2500
     .otherwise("Ultra")                          # > 2500 (transcontinental)
)

print("Distance bucket added.")
df.groupBy("distance_bucket").agg(
    F.count("*").alias("flights"),
    F.round(F.avg("DEP_DEL15") * 100, 1).alias("delay_rate_%")
).orderBy("flights").show()

In [ ]:
# Route frequency — how many times does this Origin→Dest pair fly in the dataset
# High-frequency routes have more schedule buffer and crew familiarity
route_freq = (
    df
    .groupBy("AIRPORT_CODE", "Dest")
    .agg(F.count("*").alias("route_total_flights"))
)

df = df.join(route_freq, on=["AIRPORT_CODE", "Dest"], how="left")

print("Route frequency added.")
df.select("AIRPORT_CODE", "Dest", "route_total_flights").distinct().orderBy(
    F.desc("route_total_flights")
).show(10)

In [ ]:
# Carrier historical delay rate (excluding 2020)
carrier_delay_rate = (
    df
    .filter(F.col("year") != 2020)
    .filter(F.col("DEP_DEL15").isNotNull())
    .groupBy("Reporting_Airline")
    .agg(
        F.round(F.avg("DEP_DEL15"), 4).alias("carrier_hist_delay_rate"),
        F.count("*").alias("carrier_total_flights")
    )
)

df = df.join(carrier_delay_rate, on="Reporting_Airline", how="left")

print("Carrier delay rate added (excludes 2020).")
carrier_delay_rate.orderBy(F.desc("carrier_hist_delay_rate")).show()

---
## 8. Clean Up & Final Dataset

In [ ]:
# Drop rows where target label is null — can't train on these
df_final = df.filter(F.col("DEP_DEL15").isNotNull())

# Drop rows where FL_DATE is null
df_final = df_final.filter(F.col("FL_DATE").isNotNull())

print(f"Rows before cleaning : {df.count():,}")
print(f"Rows after cleaning  : {df_final.count():,}")
print(f"Dropped              : {df.count() - df_final.count():,}")

In [ ]:
# Select final feature columns in clean order
FINAL_COLUMNS = [
    # IDs
    "FL_DATE", "AIRPORT_CODE", "Dest", "Reporting_Airline",

    # Target
    "DEP_DEL15",

    # Raw delay info (keep for reference, not used as model features)
    "DepDelay", "DepDelayMinutes", "ArrDelay", "ArrDelayMinutes", "ARR_DEL15",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
    "Cancelled", "CancellationCode",

    # Time features
    "year", "month", "day_of_month", "day_of_week", "week_of_year", "quarter",
    "dep_hour", "time_of_day", "season", "is_weekend", "is_holiday_period",

    # COVID / anomaly
    "is_covid_year", "monthly_flights", "flight_volume_index",

    # Raw weather
    "avg_temp_c", "min_temp_c", "max_temp_c", "avg_dew_point_c",
    "avg_visibility_km", "min_visibility_km",
    "avg_wind_ms", "max_wind_ms",
    "total_precip_mm", "max_precip_mm",
    "avg_pressure_hpa", "avg_ceiling_m", "min_ceiling_m",

    # Engineered weather
    "is_freezing", "is_extreme_heat", "temp_range_c",
    "wind_category", "is_high_wind",
    "visibility_category", "is_low_visibility",
    "precip_category", "is_precipitation",
    "is_low_ceiling", "weather_severity_score",

    # Airport features
    "is_hub", "airport_hist_delay_rate", "airport_hist_avg_delay_mins",
    "daily_flight_count", "congestion_index",

    # Flight / route features
    "Distance", "distance_bucket",
    "route_total_flights",
    "carrier_hist_delay_rate", "carrier_total_flights",
]

# Keep only columns that exist in df_final (guard against any missing)
existing = set(df_final.columns)
final_cols = [c for c in FINAL_COLUMNS if c in existing]
missing    = [c for c in FINAL_COLUMNS if c not in existing]

if missing:
    print(f"⚠ Columns not found (skipped): {missing}")

df_final = df_final.select(final_cols)

print(f"Final feature count : {len(final_cols)}")
df_final.printSchema()

In [ ]:
# Quick feature sanity check
df_final.select(
    "FL_DATE", "AIRPORT_CODE", "DEP_DEL15",
    "season", "time_of_day", "is_weekend", "is_holiday_period",
    "is_covid_year", "flight_volume_index",
    "weather_severity_score", "wind_category",
    "is_hub", "congestion_index", "distance_bucket"
).show(10, truncate=False)

In [ ]:
# Class balance check — important for model choice
total = df_final.count()
delayed = df_final.filter(F.col("DEP_DEL15") == 1).count()
on_time = total - delayed

print("=" * 45)
print("  CLASS BALANCE CHECK")
print("=" * 45)
print(f"  Total flights : {total:,}")
print(f"  On-time (0)   : {on_time:,}  ({on_time/total*100:.1f}%)")
print(f"  Delayed (1)   : {delayed:,}  ({delayed/total*100:.1f}%)")
print("=" * 45)
if delayed / total < 0.3:
    print("  ⚠ Class imbalance detected.")
    print("  → Use class weights or oversample in Phase 6.")
else:
    print("  ✓ Classes reasonably balanced.")

---
## 9. Save Final Feature Dataset

In [ ]:
(
    df_final
    .write
    .mode("overwrite")
    .parquet(PARQUET_OUT)
)

print(f"Saved to: {PARQUET_OUT}")

In [ ]:
# Verify
verify = spark.read.parquet(PARQUET_OUT)
print(f"Verified rows    : {verify.count():,}")
print(f"Verified columns : {len(verify.columns)}")

In [ ]:
print("=" * 55)
print("  PHASE 5 COMPLETE")
print("=" * 55)
print(f"  Output : {PARQUET_OUT}")
print(f"  Rows   : {total:,}")
print(f"  Features: {len(final_cols)}")
print("")
print("  2020 handling:")
print("  - Included in dataset")
print("  - Flagged via is_covid_year = 1")
print("  - flight_volume_index captures traffic collapse")
print("  - Airport/carrier baselines computed on 2015-2019, 2021-2024")
print("")
print("  Next → Phase 6: Model Training (Spark MLlib)")
print("  - Train/test split (stratified by year)")
print("  - Baseline: Logistic Regression")
print("  - Main: Random Forest, Gradient Boosted Trees")
print("  - Evaluate: AUC-ROC, F1, Precision, Recall")
print("=" * 55)

spark.stop()